# D1: generate the Synthea population

Generates 5,000 synthetic patients with Synthea, exports four CSV tables (patients, conditions, medications, procedures), checks their integrity and counts distinct patients per condition code. Target selection and the knowledge graph are built in later steps.

Runtime: Kaggle notebook, CPU only, internet on, JDK 17 or newer.

Settings: population seed, clinician seed, reference date and end date are all fixed to 20261104, and the Synthea commit is recorded in `metadata.json`. `-o false` turns off population overflow so that exactly 5,000 patients are exported; this changes Synthea's default behavior and is reported in the paper. Row order can vary with several threads, so `metadata.json` stores the byte hashes of this run.

References: [Synthea](https://github.com/synthetichealth/synthea), [recreating a dataset](https://github.com/synthetichealth/synthea/wiki/Recreating-a-Dataset), [CSV data dictionary](https://github.com/synthetichealth/synthea/wiki/CSV-File-Data-Dictionary).

In [ ]:
from pathlib import Path
import subprocess, sys, json, shutil, hashlib, zipfile, platform, time
import pandas as pd

ROOT = Path('/kaggle/working/synthea_d1')
REPO = ROOT / 'synthea'
RAW = ROOT / 'data/raw/synthea_seed20261104'
REPORT = ROOT / 'data'
for p in (ROOT, RAW, ROOT/'scripts', ROOT/'configs', ROOT/'logs', ROOT/'tmp', REPORT):
    p.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('Kaggle environment:', Path('/kaggle/working').exists())
java = subprocess.run(['java','-version'], capture_output=True, text=True)
javac = subprocess.run(['javac','-version'], capture_output=True, text=True)
print('Java:', (java.stderr or java.stdout).splitlines()[0] if java.returncode == 0 else 'NOT FOUND')
print('Javac:', javac.stdout.strip() if javac.returncode == 0 else 'NOT FOUND')
if java.returncode != 0 or javac.returncode != 0:
    raise RuntimeError('A JDK 17 or newer is required (a JRE is not enough).')

## Clone Synthea and record the commit

A fresh clone takes the default branch. Pin the commit printed here to reproduce the run later.

In [ ]:
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/synthetichealth/synthea.git', str(REPO)], check=True)
commit = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('SYNTHEA_COMMIT =', commit)
(ROOT/'configs/synthea_commit.txt').write_text(commit + '\n')

## CSV-only configuration

The override file enables only the four CSV tables. `exporter.years_of_history = 0` keeps the full history instead of filtering old diagnoses.

In [ ]:
CONFIG = ROOT / 'configs/synthea.properties'
CONFIG.write_text('# Synthea overrides: CSV export only\nexporter.csv.export = true\nexporter.csv.append_mode = false\nexporter.csv.folder_per_run = false\nexporter.csv.included_files = patients.csv,conditions.csv,medications.csv,procedures.csv\nexporter.years_of_history = 0\nexporter.fhir.export = false\nexporter.fhir_stu3.export = false\nexporter.fhir_dstu2.export = false\nexporter.ccda.export = false\nexporter.json.export = false\nexporter.metadata.export = false\ngenerate.log_patients.detail = none\ngenerate.thread_pool_size = 2\n', encoding='utf-8')
print(CONFIG.read_text())

## Generation script

In [ ]:
SCRIPT = ROOT/'scripts/generate_synthea.sh'
SCRIPT.write_text('#!/usr/bin/env bash\n# Generate a Synthea population with fixed seeds.\n# Usage: bash generate_synthea.sh [population] [output_base_directory]\nset -euo pipefail\nROOT="${SYNTHEA_D1_ROOT:-/kaggle/working/synthea_d1}"\nREPO="$ROOT/synthea"\nCONF="$ROOT/configs/synthea.properties"\nPOP="${1:-5000}"\nOUT="${2:-$ROOT/tmp/full}"\nif [[ ! -f "$CONF" || ! -d "$REPO/.git" ]]; then\n  echo "Expected Synthea clone at $REPO and config at $CONF" >&2\n  exit 2\nfi\nmkdir -p "$OUT"\ncd "$REPO"\n# -o false disables overflow, so exactly POP patients are exported.\n# The output override comes after -c so that it takes precedence.\nbash ./run_synthea \\\n  -c "$CONF" \\\n  "--exporter.baseDirectory=$OUT" \\\n  -s 20261104 -cs 20261104 \\\n  -r 20261104 -e 20261104 \\\n  -p "$POP" -o false Massachusetts\n', encoding='utf-8')
SCRIPT.chmod(0o755)
print('Created', SCRIPT)

## Smoke test with 12 patients

Checks the build, the Java setup and the exporter. The output goes to `tmp/smoke` and is not part of the final population.

In [ ]:
def generate(population: int, output_base: Path, log_path: Path):
    output_base.mkdir(parents=True, exist_ok=True)
    cmd = ['bash', str(SCRIPT), str(population), str(output_base)]
    print('Running:', ' '.join(cmd))
    start = time.monotonic()
    with log_path.open('w', encoding='utf-8') as f:
        completed = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True)
    tail = log_path.read_text(errors='replace').splitlines()[-22:]
    print('Exit code:', completed.returncode, '| minutes elapsed:', round((time.monotonic()-start)/60, 2))
    print('\n'.join(tail))
    if completed.returncode != 0:
        raise RuntimeError(f'Synthea failed: inspect {log_path}; last lines shown above.')
    csv_dir = output_base/'csv'
    for name in ('patients.csv','conditions.csv','medications.csv','procedures.csv'):
        assert (csv_dir/name).exists(), f'Missing {csv_dir/name}'
    return csv_dir

smoke_dir = ROOT/'tmp/smoke'
smoke_log = ROOT/'logs/smoke.log'
if not (smoke_dir/'csv/patients.csv').exists():
    generate(12, smoke_dir, smoke_log)
else:
    print('Smoke CSV already exists; skipping smoke rerun.')

## Full run with 5,000 patients

Same settings as the smoke test. The cell is skipped if the four raw CSV files already exist.

In [ ]:
FULL_DIR = ROOT/'tmp/full'
required = ('patients.csv','conditions.csv','medications.csv','procedures.csv')
if not all((RAW/name).exists() for name in required):
    generated_csv_dir = generate(5000, FULL_DIR, ROOT/'logs/generation_5000.log')
    # Copy only after generation finished without error.
    for name in required:
        shutil.copy2(generated_csv_dir/name, RAW/name)
    print('Copied raw CSVs to:', RAW)
else:
    print('Raw CSVs already exist; generation was not repeated.')
for name in required:
    path = RAW/name
    print(f'{name:20s} {path.stat().st_size/1024/1024:>8.2f} MiB')

## Screening script

Checks for 5,000 unique patient IDs and no orphan clinical rows, and counts distinct patients per condition code. Writes `disease_counts.csv`, `candidate_screening.csv` and `quality_report.json`.

A code with 60 patients does not guarantee 60 single-target patients; that is checked in `reproduce_selection.py`. Medication and procedure counts can come from unrelated conditions. Only REASONCODE-linked counts indicate an explicit association, and they may be incomplete.

In [ ]:
ANALYSIS_SCRIPT = ROOT/'scripts/analyze_synthea.py'
ANALYSIS_SCRIPT.write_text('#!/usr/bin/env python3\n"""Integrity checks and candidate condition-code screening for the D1 Synthea export.\n\nThis is a first screen only. Patient-level splits and the final target selection\nare done in reproduce_selection.py.\n"""\nfrom __future__ import annotations\nimport argparse\nimport json\nfrom pathlib import Path\nimport pandas as pd\n\nREQUIRED = ("patients.csv", "conditions.csv", "medications.csv", "procedures.csv")\n\n\ndef load_csv(path: Path) -> pd.DataFrame:\n    if not path.exists() or path.stat().st_size == 0:\n        raise FileNotFoundError(f"Missing/empty expected CSV: {path}")\n    df = pd.read_csv(path, dtype="string", low_memory=False)\n    df.columns = [str(c).upper().strip() for c in df.columns]\n    return df\n\n\ndef normalize_str_column(df: pd.DataFrame, col: str):\n    if col in df.columns:\n        df[col] = df[col].fillna("").astype(str).str.strip()\n\n\ndef reason_patient_counts(events: pd.DataFrame, diagnosed: pd.DataFrame) -> pd.Series:\n    if "REASONCODE" not in events.columns:\n        return pd.Series(dtype="int64")\n    data = events.loc[events["REASONCODE"].ne(""), ["PATIENT", "REASONCODE"]].copy()\n    data = data.rename(columns={"REASONCODE": "CODE"}).drop_duplicates()\n    if data.empty:\n        return pd.Series(dtype="int64")\n    # Match on exact (patient, code) pairs, not on similar descriptions.\n    matched = diagnosed.merge(data, on=["PATIENT", "CODE"], how="inner")\n    return matched.groupby("CODE")["PATIENT"].nunique()\n\n\ndef main(raw: Path, out: Path, expected: int):\n    out.mkdir(parents=True, exist_ok=True)\n    patients, cond, meds, proc = [load_csv(raw / f) for f in REQUIRED]\n    for label, df, cols in [\n        ("patients", patients, ["ID"]),\n        ("conditions", cond, ["PATIENT", "CODE", "DESCRIPTION"]),\n        ("medications", meds, ["PATIENT"]),\n        ("procedures", proc, ["PATIENT"]),\n    ]:\n        missing = set(cols) - set(df.columns)\n        if missing:\n            raise ValueError(f"{label}.csv missing columns {missing}; columns are {list(df.columns)}")\n        for c in ["ID", "PATIENT", "CODE", "DESCRIPTION", "REASONCODE"]:\n            normalize_str_column(df, c)\n\n    patient_ids = set(patients["ID"])\n    n_patients = patients["ID"].nunique()\n    duplicate_ids = int(patients["ID"].duplicated().sum())\n    clinical = {"conditions": cond, "medications": meds, "procedures": proc}\n    orphans = {name: int((~df["PATIENT"].isin(patient_ids)).sum())\n               for name, df in clinical.items()}\n    missing_codes = int(cond["CODE"].eq("").sum())\n    if n_patients != expected or duplicate_ids or any(orphans.values()) or missing_codes:\n        raise AssertionError(\n            f"Data quality FAIL: unique_patients={n_patients} expected={expected}; "\n            f"duplicate_ID={duplicate_ids}; orphan_rows={orphans}; missing_condition_codes={missing_codes}"\n        )\n\n    # One patient can have multiple condition events; count patients once per disease code.\n    diagnosed = cond.loc[cond["CODE"].ne(""), ["PATIENT", "CODE"]].drop_duplicates()\n    counts = diagnosed.groupby("CODE")["PATIENT"].nunique().rename("n_patients")\n    events = cond.groupby("CODE").size().rename("n_condition_events")\n    descriptions = (\n        cond.loc[cond["CODE"].ne(""), ["CODE", "DESCRIPTION"]]\n        .drop_duplicates()\n        .groupby("CODE")["DESCRIPTION"].agg(lambda x: " | ".join(sorted(x.unique())[:3]))\n        .rename("description")\n    )\n    medicated = set(meds["PATIENT"])\n    procedured = set(proc["PATIENT"])\n    diagnosed["med_any"] = diagnosed["PATIENT"].isin(medicated)\n    diagnosed["proc_any"] = diagnosed["PATIENT"].isin(procedured)\n    med_any = diagnosed.groupby("CODE")["med_any"].sum().rename("n_patients_with_any_medication")\n    proc_any = diagnosed.groupby("CODE")["proc_any"].sum().rename("n_patients_with_any_procedure")\n    med_direct = reason_patient_counts(meds, diagnosed[["PATIENT", "CODE"]]).rename("n_patients_with_reason_linked_medication")\n    proc_direct = reason_patient_counts(proc, diagnosed[["PATIENT", "CODE"]]).rename("n_patients_with_reason_linked_procedure")\n\n    profile = pd.concat([descriptions, counts, events, med_any, proc_any, med_direct, proc_direct], axis=1).reset_index()\n    for col in profile.columns:\n        if col.startswith("n_"):\n            profile[col] = profile[col].fillna(0).astype(int)\n    profile["pct_patients"] = (100 * profile["n_patients"] / n_patients).round(2)\n    profile["pct_with_any_medication"] = (100 * profile["n_patients_with_any_medication"] / profile["n_patients"]).round(2)\n    profile["pct_with_any_procedure"] = (100 * profile["n_patients_with_any_procedure"] / profile["n_patients"]).round(2)\n    profile = profile.sort_values(["n_patients", "CODE"], ascending=[False, True]).reset_index(drop=True)\n    profile.to_csv(out / "disease_counts.csv", index=False)\n\n    # 60 patients per code is necessary but not sufficient for six single-target classes.\n    screen = profile.loc[profile["n_patients"] >= 60].copy()\n    screen["passes_raw_60_patient_screen"] = True\n    # Co-diagnoses shrink the single-target pool, so record the largest overlap per code.\n    patient_by_code = {code: set(group["PATIENT"]) for code, group in diagnosed.groupby("CODE")}\n    code_list = screen["CODE"].tolist()\n    max_overlap_fraction, overlap_partner = [], []\n    for code in code_list:\n        pool = patient_by_code[code]\n        other_code, frac = "", 0.0\n        for candidate in code_list:\n            if candidate == code:\n                continue\n            f = len(pool & patient_by_code[candidate]) / len(pool)\n            if f > frac:\n                other_code, frac = candidate, f\n        max_overlap_fraction.append(round(frac, 4))\n        overlap_partner.append(other_code)\n    screen["max_overlap_fraction_with_other_screened_code"] = max_overlap_fraction\n    screen["most_overlapping_screened_code"] = overlap_partner\n    screen["single_target_count"] = pd.NA  # filled in after the six codes are chosen\n    screen["review_status"] = "MANUAL_REVIEW_REQUIRED"\n    screen.to_csv(out / "candidate_screening.csv", index=False)\n\n    summary = {\n        "screening_not_final_selection": True,\n        "n_patients": int(n_patients),\n        "n_condition_events": int(len(cond)),\n        "n_medication_events": int(len(meds)),\n        "n_procedure_events": int(len(proc)),\n        "n_disease_codes": int(profile.shape[0]),\n        "n_raw_candidates_ge_60": int(screen.shape[0]),\n        "n_patients_with_any_condition": int(cond["PATIENT"].nunique()),\n        "orphan_rows": orphans,\n        "duplicate_patient_ids": duplicate_ids,\n        "conditions_missing_code": missing_codes,\n        "medication_reasoncode_column_present": "REASONCODE" in meds.columns,\n        "procedure_reasoncode_column_present": "REASONCODE" in proc.columns,\n        "preliminary_candidate_gate": "PASS" if len(screen) >= 6 else "FAIL_REVIEW_POPULATION_OR_SELECTION",\n        "note": "60 raw patients/code is not equivalent to 60 patients exclusively in one of the six final target classes. Check on D2."\n    }\n    (out / "quality_report.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")\n    print(json.dumps(summary, indent=2, ensure_ascii=False))\n    print("\\nTop 20 disease concepts (code-level, distinct patients):")\n    print(profile[["CODE", "description", "n_patients", "n_condition_events", "n_patients_with_reason_linked_medication", "n_patients_with_reason_linked_procedure"]].head(20).to_string(index=False, max_colwidth=45))\n    print("\\nSaved outputs:", *(str(p) for p in sorted(out.glob("*"))), sep="\\n  - ")\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--raw", type=Path, required=True)\n    parser.add_argument("--out", type=Path, required=True)\n    parser.add_argument("--expected", type=int, default=5000)\n    args = parser.parse_args()\n    main(args.raw, args.out, args.expected)\n', encoding='utf-8')
print('Created', ANALYSIS_SCRIPT, '| lines:', len(ANALYSIS_SCRIPT.read_text().splitlines()))

## Integrity check and screening

In [ ]:
from IPython.display import display

cmd = [sys.executable, str(ANALYSIS_SCRIPT), '--raw', str(RAW), '--out', str(REPORT), '--expected', '5000']
subprocess.run(cmd, check=True)
print('\nTop 30 screened candidates:')
screen = pd.read_csv(REPORT/'candidate_screening.csv')
cols = ['CODE','description','n_patients','n_patients_with_reason_linked_medication',
        'n_patients_with_reason_linked_procedure','max_overlap_fraction_with_other_screened_code']
display(screen[cols].head(30))

## Provenance and archive

`metadata.json` records the Synthea commit, the settings, the config hash and the hash of each CSV. The archive contains the generated data and the scripts, not the Synthea repository.

In [ ]:
def sha256(path: Path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

metadata = {
    'project': 'ARoG-style KGQA disease-class leakage audit, D1 data generation',
    'synthea_repository': 'https://github.com/synthetichealth/synthea',
    'synthea_commit': subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip(),
    'population_requested': 5000,
    'seed': 20261104, 'clinician_seed': 20261104,
    'reference_date': '2026-11-04', 'end_date': '2026-11-04',
    'location': 'Massachusetts, USA', 'overflow_population': False,
    'csv_included_files': list(required),
    'config_sha256': sha256(CONFIG),
    'java_version': (subprocess.run(['java','-version'],capture_output=True,text=True).stderr.splitlines() or ['unknown'])[0],
    'csv_sha256': {name: sha256(RAW/name) for name in required},
    'note': 'Row order can vary with multiple threads; this file records byte hashes of this run.'
}
(ROOT/'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
ZIP = Path('/kaggle/working/Synthea_D1_20261104_outputs.zip')
include = [*RAW.glob('*.csv'), *REPORT.glob('*.csv'), REPORT/'quality_report.json',
           CONFIG, ROOT/'configs/synthea_commit.txt', SCRIPT, ANALYSIS_SCRIPT, ROOT/'metadata.json']
include += [p for p in (ROOT/'logs').glob('*.log') if p.exists()]
with zipfile.ZipFile(ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for f in include:
        archive.write(f, arcname=str(f.relative_to(ROOT)))
print('Archive:', ZIP)
print('ZIP size MiB:', round(ZIP.stat().st_size/1024**2,2))
print('Final source CSV count:', len(list(RAW.glob('*.csv'))))

## Acceptance criteria

- `quality_report.json` reports 5,000 patients, no duplicate IDs, no orphan clinical rows and no empty condition codes.
- `candidate_screening.csv` lists at least six codes with 60 or more patients.
- The raw CSV files stay unchanged after this step.